# Balancing

In this notebook you will learn how to

- find products and columns that do not balance
- set targets, tolerances and locks
- balance products and columns in turn
- check the result

**The change we make to the data is invented.** It is only there to have something to balance.

## Load the data

See [01_getting_started](01_getting_started.ipynb) for an explanation of these steps.

In [ ]:
import sys
from pathlib import Path

# Location of sutlab. Absolute, so that the notebook can be copied to another folder
sutlab_folder = Path(r"C:\Users\DstMove\Desktop\claude\projects\sutlab")
sys.path.insert(0, str(sutlab_folder))

import sutlab

data_folder = sutlab_folder / "docs" / "examples" / "data"
config_folder = sutlab_folder / "docs" / "examples" / "config"
years = [2016, 2017, 2018, 2019]

metadata = sutlab.load_metadata_from_excel(
    data_folder / "ta_columns.xlsx",
    data_folder / "ta_classifications.xlsx",
)

current_paths = sutlab.make_final_annual_sut_paths(years, "current_year")
sut = sutlab.load_sut_from_separated_nr_legacy_fwf(years, current_paths, metadata, "current_year")

## Make an imbalance

Suppose a new source says that household consumption of wine in 2019 is higher than in the table. The difference is in an Excel file, which we load and add. [05_adjusting_and_comparing](05_adjusting_and_comparing.ipynb) explains how this works.

In [ ]:
change = sutlab.load_sut_from_combined_excel(
    config_folder / "revision_households.xlsx",
    metadata,
    "current_year",
)
sut_changed = sut.adjust_add_sut(change)
change.use

## Set up

Balancing works on one year at a time, chosen with `set_balancing_id`. The other years are not changed.

In [ ]:
sut_changed = sut_changed.set_balancing_id(2019)

**Targets** are the column totals we want to hit. Normally they come from a file (see the functions that start with `sutlab.load_balancing_targets_`). Here we use the totals from before our change.

`compute_totals` sums over products, and so gives one row per transaction and category.

In [ ]:
totals_before = sut.filter_rows(ids=2019).compute_totals(["transaction", "category"])
targets = sutlab.load_balancing_targets_from_dataframe(totals_before, metadata)

sut_changed = sut_changed.set_balancing_targets(targets)
targets.use.head()

**Tolerances and locks** are kept in two Excel files in the folder `config`. The values in them are invented. You can open the files in Excel to see how they look.

In [ ]:
config = sutlab.load_balancing_config_from_excel(
    metadata,
    tolerances_path=config_folder / "balancing_tolerances.xlsx",
    locks_path=config_folder / "balancing_locks.xlsx",
)
sut_changed = sut_changed.set_balancing_config(config)

Tolerances say how far from its target a column may be. The sheet `transactions` has a relative and an absolute limit for each transaction, and the stricter of the two applies. Single columns can be given their own limits in a sheet called `categories`.

In [ ]:
config.target_tolerances.transactions.head()

Locks protect cells from being changed. We trust the new figure for wine, so that cell is locked in the sheet `cells`. Use the sheets `products`, `transactions`, `categories` and `price_layers` to lock more at a time. Sheets you do not need can be left out.

In [ ]:
config.locks.cells

## Find the imbalances

Use is now larger than supply for wine. We only look at a range of drinks, since that is where we made the change.

In [ ]:
drinks = "V220400:V220800"

sut_changed.inspect_unbalanced_products(drinks, ids=2019).imbalances

The column for household consumption of wine and spirits is also too high. A violation is the part of the difference that is outside the tolerance.

In [ ]:
sut_changed.inspect_unbalanced_targets(ids=2019).use_categories_violations

## Balance products

`balance_products_use` scales the use of each product so that it matches supply. All price columns of a row are scaled by the same factor, which keeps the rates for margins and taxes unchanged.

We choose which cells may be changed: household consumption of wine and spirits (`02112`) and intermediate consumption in restaurants (`560000`).

In [ ]:
adjust_transactions = ["2000", "3110"]
adjust_categories = ["02112", "560000"]

sut_balanced = sut_changed.balance_products_use(
    products=drinks,
    adjust_transactions=adjust_transactions,
    adjust_categories=adjust_categories,
)
sut_balanced.inspect_unbalanced_products(drinks, ids=2019).imbalances

The products balance. The locked cell has not changed, so the adjustment was made in restaurants instead. Now that column is too low, and the column for households is still too high.

In [ ]:
sut_balanced.inspect_unbalanced_targets(ids=2019).use_categories_violations

## Balance columns

`balance_columns` does the opposite: it scales the products in a column so that the total matches the target.

In [ ]:
sut_balanced = sut_balanced.balance_columns(
    transactions=adjust_transactions,
    categories=adjust_categories,
    adjust_products=drinks,
)
sut_balanced.inspect_unbalanced_products(drinks, ids=2019).imbalances

The columns now hit their targets, but the products are out of balance again, though wine by less than before. If we repeat the two steps, the differences become smaller and smaller. We finish with the products, so they balance exactly while the columns are within their tolerance.

In [ ]:
for round_number in range(10):
    sut_balanced = sut_balanced.balance_products_use(
        products=drinks,
        adjust_transactions=adjust_transactions,
        adjust_categories=adjust_categories,
    )
    sut_balanced = sut_balanced.balance_columns(
        transactions=adjust_transactions,
        categories=adjust_categories,
        adjust_products=drinks,
    )

sut_balanced = sut_balanced.balance_products_use(
    products=drinks,
    adjust_transactions=adjust_transactions,
    adjust_categories=adjust_categories,
)

## Check the result

Products:

In [ ]:
sut_balanced.inspect_unbalanced_products(drinks, ids=2019).imbalances

Columns outside their tolerance:

In [ ]:
sut_balanced.inspect_unbalanced_targets(ids=2019).use_categories_violations

Both tables are empty, so we are done with these drinks. Finally, we compare with the table we started from to see what it took.

In [ ]:
comparison = sut_balanced.inspect_sut_comparison(sut, ids=2019)
comparison.use_purchasers

**Note:** changing use also changes the total margins and taxes. They have to match the supply of margins and the known tax revenue, which we have not looked at here.

---
Next: [05_adjusting_and_comparing](05_adjusting_and_comparing.ipynb)